# RQ2 — Temporal Modelling & Adaptive Alarm Triggering

Compara estrategias estrictamente causales de disparo de alarma sobre la secuencia cronológica de mensajes por usuario (`date`, desempate por `id_message`):
1. **Baseline:** Cumulative Mean Pooling + umbral fijo $\tau$.
2. **Moving Window / Recent Mean:** últimos $k \in \{3, 5, 10\}$ mensajes.
3. **EWMA:** $\lambda \in \{0.1, 0.2, 0.3, 0.5, 0.7\}$.
4. **Consecutive Threshold:** $r \in \{2, 3, 4\}$ mensajes consecutivos con $p_t \ge \tau$.
5. **Alarma adaptativa aprendida:** **Logistic Regression** y **Ridge Regression** sobre características temporales del prefijo (`mean_cum`, `max_cum`, `recent_mean_5`, `slope_recent`, `std_cum`, `diff_recent5_hist`, `prop_above_05`, `message_order`) con validación cruzada agrupada por `subject_id` en `dev` y ponderación inversa a la longitud del historial.

Métricas: **Macro-F1**, **Precision**, **Recall**, **FPR**, **FNR**, **ERDE5**, **ERDE30**, latencia media/mediana hasta la alarma y porcentaje de usuarios sin alarma.

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/Urkibiris05/dm_project.git'
GIT_BRANCH = 'rq_larra'
GITHUB_TOKEN = ''

def _is_repo_root(p: Path) -> bool:
    return (p / 'particiones').is_dir() and (p / 'rq_larra' / 'rq_utils.py').is_file()

def _find_repo() -> Path | None:
    candidates = [Path('/content/dm_project'), Path.cwd().resolve(), *Path.cwd().resolve().parents]
    drive_root = Path('/content/drive/MyDrive')
    if drive_root.is_dir():
        candidates.append(drive_root / 'dm_project')
        for pattern in ('*', '*/*', '*/*/*'):
            for p in drive_root.glob(pattern):
                if p.is_dir() and p.name == 'dm_project':
                    candidates.append(p)
    return next((p for p in candidates if _is_repo_root(p)), None)

PROJECT_ROOT = _find_repo()
if PROJECT_ROOT is None and IN_COLAB:
    clone_dir = Path('/content/dm_project')
    clone_url = REPO_URL.replace('https://', f'https://{GITHUB_TOKEN}@') if GITHUB_TOKEN else REPO_URL
    if not clone_dir.exists():
        subprocess.run(['git', 'clone', '-b', GIT_BRANCH, clone_url, str(clone_dir)], text=True, check=False)
    PROJECT_ROOT = _find_repo()

if PROJECT_ROOT is None:
    raise FileNotFoundError('No se encontró dm_project con particiones/ y rq_larra/rq_utils.py. Sube el repo a Drive o haz push a GitHub.')

if str(PROJECT_ROOT / 'rq_larra') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'rq_larra'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rq_utils

DRIVE_STORAGE_DIR = '/content/drive/MyDrive/dm_project_artifacts' if IN_COLAB else None
STORAGE_ROOT = rq_utils.resolve_storage_root(PROJECT_ROOT, DRIVE_STORAGE_DIR)

backup_base = STORAGE_ROOT / 'rq_larra_backup' / 'csv' / 'rq_baseline'
local_base = PROJECT_ROOT / 'rq_larra' / 'csv' / 'rq_baseline'
if not (local_base / 'latest_baseline.json').is_file() and (backup_base / 'latest_baseline.json').is_file():
    shutil.copytree(backup_base, local_base, dirs_exist_ok=True)
    print('Restaurado baseline desde Drive:', backup_base)

BASELINE_RUN_ID = None
pointer = rq_utils.load_latest_baseline_pointer(PROJECT_ROOT)
if BASELINE_RUN_ID is None:
    BASELINE_RUN_ID = pointer['run_id']

BASELINE_CSV_DIR = PROJECT_ROOT / 'rq_larra' / 'csv' / 'rq_baseline' / BASELINE_RUN_ID
DEV_MSG_CSV = BASELINE_CSV_DIR / 'dev_message_predictions.csv'
if not DEV_MSG_CSV.is_file():
    raise FileNotFoundError(f'Falta {DEV_MSG_CSV}. Ejecuta primero 00_baseline_relabeling.ipynb.')

RUN_ID = rq_utils.make_run_id('rq2')
CSV_DIR = PROJECT_ROOT / 'rq_larra' / 'csv' / 'rq2' / RUN_ID
FIG_DIR = PROJECT_ROOT / 'rq_larra' / 'figuras' / 'rq2' / RUN_ID
CSV_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

dev_msgs = pd.read_csv(DEV_MSG_CSV, dtype={'subject_id': str, 'id_message': str})
print(f'RUN_ID: {RUN_ID} | Baseline: {BASELINE_RUN_ID} | Filas dev: {len(dev_msgs)}')
dev_msgs.head()

In [ ]:
# 1) Construir características temporales causales por prefijo + modelos adaptativos OOF en dev
WINDOWS = (3, 5, 10)
EWMA_LAMBDAS = (0.1, 0.2, 0.3, 0.5, 0.7)
TEMPORAL_FEATURES = [
    'mean_cum',
    'max_cum',
    'recent_mean_5',
    'slope_recent',
    'std_cum',
    'diff_recent5_hist',
    'prop_above_05',
    'message_order',
]

prefix_frames = []
fitted_temporal_models = {}

for (disorder, condition, seed), grp in dev_msgs.groupby(['disorder', 'condition', 'seed']):
    pref = rq_utils.build_prefix_features_df(grp, moving_windows=WINDOWS, ewma_lambdas=EWMA_LAMBDAS)
    pref['score_logreg'], logreg_full = rq_utils.fit_predict_oof_temporal_alarm(
        pref, TEMPORAL_FEATURES, model_type='logistic', n_splits=5
    )
    pref['score_ridge'], ridge_full = rq_utils.fit_predict_oof_temporal_alarm(
        pref, TEMPORAL_FEATURES, model_type='ridge', n_splits=5
    )
    fitted_temporal_models[(disorder, condition, int(seed))] = {
        'logistic': logreg_full,
        'ridge': ridge_full,
    }
    prefix_frames.append(pref)

dev_prefixes = pd.concat(prefix_frames, ignore_index=True)
print('Prefijos construidos:', len(dev_prefixes))

In [ ]:
# 2) Barrido de estrategias temporales y selección de umbral en dev
PROB_GRID = np.round(np.linspace(0.20, 0.90, 15), 2)
RIDGE_GRID = np.round(np.linspace(0.10, 0.90, 17), 2)

STRATEGIES = [
    ('baseline_mean_cum', 'mean_cum', 1, PROB_GRID),
    *[ (f'window_k{k}', f'recent_mean_{k}', 1, PROB_GRID) for k in WINDOWS ],
    *[ (f'ewma_l{str(l).replace(".", "")}', f'ewma_{str(l).replace(".", "")}', 1, PROB_GRID) for l in EWMA_LAMBDAS ],
    *[ (f'consecutive_r{r}', 'prob', r, PROB_GRID) for r in (2, 3, 4) ],
    ('adaptive_logreg', 'score_logreg', 1, PROB_GRID),
    ('adaptive_ridge', 'score_ridge', 1, RIDGE_GRID),
]

sweep_rows = []
for (disorder, condition, seed), grp in dev_prefixes.groupby(['disorder', 'condition', 'seed']):
    for strat_name, score_col, consec_r, grid in STRATEGIES:
        for tau in grid:
            _, metrics = rq_utils.evaluate_alarm_rule(
                grp, score_col=score_col, threshold=float(tau), consecutive_r=consec_r
            )
            sweep_rows.append({
                'disorder': disorder,
                'condition': condition,
                'seed': seed,
                'strategy': strat_name,
                'score_col': score_col,
                'consecutive_r': consec_r,
                'threshold': float(tau),
                **metrics,
            })

sweep_df = pd.DataFrame(sweep_rows)
sweep_df.to_csv(CSV_DIR / 'temporal_strategies_grid_seed.csv', index=False)

# Promedio sobre semillas por configuración (disorder, condition, strategy, threshold)
metric_cols = [
    'macro_f1', 'precision_pos', 'recall_pos', 'fpr', 'fnr',
    'erde5', 'erde30', 'tp_latency_msg_mean', 'tp_latency_msg_median',
    'tp_latency_frac_mean', 'pct_users_alarmed', 'pct_users_no_alarm'
]
grid_mean = (
    sweep_df.groupby(['disorder', 'condition', 'strategy', 'score_col', 'consecutive_r', 'threshold'])[metric_cols]
    .agg(['mean', 'std']).reset_index()
)
grid_mean.columns = ['_'.join(c).rstrip('_') for c in grid_mean.columns]
grid_mean.to_csv(CSV_DIR / 'temporal_strategies_grid_summary.csv', index=False)

# Seleccionar mejor umbral en dev bajo dos criterios declarados: (A) menor ERDE5, (B) mayor Macro-F1
best_erde5 = grid_mean.sort_values(['erde5_mean', 'macro_f1_mean'], ascending=[True, False]).groupby(['disorder', 'condition', 'strategy'], as_index=False).first()
best_erde5['selection_criterion'] = 'min_erde5'
best_f1 = grid_mean.sort_values(['macro_f1_mean', 'erde5_mean'], ascending=[False, True]).groupby(['disorder', 'condition', 'strategy'], as_index=False).first()
best_f1['selection_criterion'] = 'max_macro_f1'

selected_df = pd.concat([best_erde5, best_f1], ignore_index=True)
selected_df.to_csv(CSV_DIR / 'temporal_strategies_selected_dev.csv', index=False)
best_erde5[best_erde5['condition'] == 'knn_include_same_subject'][['disorder', 'strategy', 'threshold', 'macro_f1_mean', 'erde5_mean', 'erde30_mean', 'tp_latency_msg_median_mean', 'fpr_mean']]

In [ ]:
# 3) Guardar decisiones por usuario de las estrategias seleccionadas (criterio min_erde5)
user_outcomes_list = []
for _, cfg_row in best_erde5.iterrows():
    sub_pref = dev_prefixes[(dev_prefixes['disorder'] == cfg_row['disorder']) & (dev_prefixes['condition'] == cfg_row['condition'])]
    for seed, s_grp in sub_pref.groupby('seed'):
        outcomes, _ = rq_utils.evaluate_alarm_rule(
            s_grp,
            score_col=cfg_row['score_col'],
            threshold=float(cfg_row['threshold']),
            consecutive_r=int(cfg_row['consecutive_r']),
        )
        outcomes['disorder'] = cfg_row['disorder']
        outcomes['condition'] = cfg_row['condition']
        outcomes['strategy'] = cfg_row['strategy']
        outcomes['seed'] = seed
        outcomes['threshold'] = float(cfg_row['threshold'])
        user_outcomes_list.append(outcomes)

user_outcomes_df = pd.concat(user_outcomes_list, ignore_index=True)
user_outcomes_df.to_csv(CSV_DIR / 'dev_user_alarm_outcomes.csv', index=False)
print('Guardadas decisiones por usuario:', len(user_outcomes_df))

In [ ]:
# 4) Figuras RQ2: ERDE5 vs Macro-F1, latencia y comparación de estrategias
plot_df = best_erde5[best_erde5['condition'] == 'knn_include_same_subject']
shortlist = ['baseline_mean_cum', 'window_k5', 'ewma_l03', 'consecutive_r2', 'adaptive_logreg', 'adaptive_ridge']
sub_short = plot_df[plot_df['strategy'].isin(shortlist)]

# Figura 1: Trade-off Macro-F1 vs ERDE5
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, disorder in zip(axes, rq_utils.DISORDERS):
    d_sub = plot_df[plot_df['disorder'] == disorder]
    ax.scatter(d_sub['erde5_mean'], d_sub['macro_f1_mean'], alpha=0.75, s=55)
    for _, r in d_sub[d_sub['strategy'].isin(shortlist)].iterrows():
        ax.annotate(r['strategy'], (r['erde5_mean'], r['macro_f1_mean']), fontsize=7, xytext=(4, 2), textcoords='offset points')
    ax.set_title(disorder)
    ax.set_xlabel('ERDE5 (menor es mejor)')
    ax.grid(alpha=0.3)
axes[0].set_ylabel('Macro-F1 (mayor es mejor)')
plt.suptitle('RQ2: Trade-off Detección Temprana (ERDE5) vs Macro-F1 en dev')
plt.tight_layout()
plt.savefig(FIG_DIR / 'erde5_vs_macro_f1.png', dpi=150)
plt.show()

# Figura 2: Comparación directa de estrategias representativas (ERDE5 y ERDE30)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
piv_e5 = sub_short.pivot(index='strategy', columns='disorder', values='erde5_mean').reindex(shortlist)
piv_e30 = sub_short.pivot(index='strategy', columns='disorder', values='erde30_mean').reindex(shortlist)
piv_e5.plot(kind='bar', ax=axes[0])
axes[0].set_title('ERDE5 por estrategia (menor es mejor)')
axes[0].set_ylabel('ERDE5')
axes[0].grid(axis='y', alpha=0.3)
axes[0].tick_params(axis='x', rotation=25)

piv_e30.plot(kind='bar', ax=axes[1])
axes[1].set_title('ERDE30 por estrategia (menor es mejor)')
axes[1].set_ylabel('ERDE30')
axes[1].grid(axis='y', alpha=0.3)
axes[1].tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.savefig(FIG_DIR / 'erde_strategy_comparison.png', dpi=150)
plt.show()

# Figura 3: Ejemplo anónimo de evolución temporal de riesgo (1 sujeto suffer y 1 control de Depresion)
ex = dev_prefixes[(dev_prefixes['disorder'] == 'Depresion') & (dev_prefixes['condition'] == 'knn_include_same_subject') & (dev_prefixes['seed'] == 11)]
pos_subj = ex[ex['label'] == 1]['subject_id'].iloc[0]
neg_subj = ex[ex['label'] == 0]['subject_id'].iloc[0]
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, sid, title in [(axes[0], pos_subj, f'Sujeto positivo ({pos_subj})'), (axes[1], neg_subj, f'Sujeto control ({neg_subj})')]:
    s_df = ex[ex['subject_id'] == sid]
    ax.plot(s_df['message_order'], s_df['prob'], alpha=0.35, label='prob mensaje')
    ax.plot(s_df['message_order'], s_df['mean_cum'], linewidth=2, label='mean_cum')
    ax.plot(s_df['message_order'], s_df['ewma_03'], linewidth=2, label='ewma (0.3)')
    ax.plot(s_df['message_order'], s_df['score_logreg'], linewidth=2, linestyle='--', label='LogReg temporal')
    ax.set_title(title)
    ax.set_xlabel('Mensaje cronológico t')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
axes[0].set_ylabel('Puntuación de riesgo')
plt.tight_layout()
plt.savefig(FIG_DIR / 'temporal_trajectories_example.png', dpi=150)
plt.show()

In [ ]:
# 5) Registrar en index_runs.csv y respaldo en Drive
rq_utils.register_run(
    PROJECT_ROOT,
    run_id=RUN_ID,
    rq='rq2',
    stage='dev_analysis',
    status='completed',
    notebook='rq_larra/ipynb/rq2.ipynb',
    seed='11,23,37,51,79',
    disorder='all',
    embeddings='beto_sts_768',
    artifacts_dir=f'rq_larra/csv/rq2/{RUN_ID}',
    notes=f'baseline_run={BASELINE_RUN_ID}; mean_cum, window, ewma, consecutive, logreg, ridge + ERDE5/30',
)
if IN_COLAB and not str(PROJECT_ROOT).startswith('/content/drive'):
    backup_root = STORAGE_ROOT / 'rq_larra_backup'
    shutil.copytree(CSV_DIR, backup_root / 'csv' / 'rq2' / RUN_ID, dirs_exist_ok=True)
    shutil.copytree(FIG_DIR, backup_root / 'figuras' / 'rq2' / RUN_ID, dirs_exist_ok=True)
    shutil.copy2(PROJECT_ROOT / 'rq_larra' / 'csv' / 'index_runs.csv', backup_root / 'csv' / 'index_runs.csv')
    print('Backup sincronizado en Drive:', backup_root)

In [ ]:
# 6) [SOLO CON PROTOCOLO CONGELADO] Evaluación final sobre test con estrategias/umbrales elegidos en dev
RUN_FROZEN_TEST = False

if RUN_FROZEN_TEST:
    test_msgs = pd.read_csv(BASELINE_CSV_DIR / 'test_message_predictions.csv', dtype={'subject_id': str, 'id_message': str})
    test_pref_list = []
    for (disorder, condition, seed), grp in test_msgs.groupby(['disorder', 'condition', 'seed']):
        pref = rq_utils.build_prefix_features_df(grp, moving_windows=WINDOWS, ewma_lambdas=EWMA_LAMBDAS)
        models = fitted_temporal_models[(disorder, condition, int(seed))]
        x_test = pref[TEMPORAL_FEATURES].to_numpy(dtype=np.float64)
        pref['score_logreg'] = models['logistic']['estimator'].predict_proba(
            models['logistic']['scaler'].transform(x_test)
        )[:, 1]
        pref['score_ridge'] = models['ridge']['estimator'].predict(
            models['ridge']['scaler'].transform(x_test)
        )
        test_pref_list.append(pref)
    test_prefixes = pd.concat(test_pref_list, ignore_index=True)

    test_rows = []
    for _, cfg_row in best_erde5.iterrows():
        sub_t = test_prefixes[(test_prefixes['disorder'] == cfg_row['disorder']) & (test_prefixes['condition'] == cfg_row['condition'])]
        for seed, s_grp in sub_t.groupby('seed'):
            _, metrics = rq_utils.evaluate_alarm_rule(
                s_grp,
                score_col=cfg_row['score_col'],
                threshold=float(cfg_row['threshold']),
                consecutive_r=int(cfg_row['consecutive_r']),
            )
            test_rows.append({
                'disorder': cfg_row['disorder'],
                'condition': cfg_row['condition'],
                'strategy': cfg_row['strategy'],
                'seed': seed,
                'threshold': float(cfg_row['threshold']),
                **metrics,
            })
    test_rq2 = pd.DataFrame(test_rows)
    test_rq2.to_csv(CSV_DIR / 'test_frozen_rq2_metrics.csv', index=False)
    print('Evaluación congelada RQ2 guardada en test_frozen_rq2_metrics.csv')